# Organized research notebook

Cell source is retained from the supplied research archive. Saved outputs and credentials have been removed. Run the setup cell first. Check model, SAE, file paths, and run settings in the cells before starting a GPU experiment. This notebook has been checked statically, not rerun against the paper results. See `docs/RELEASE_REVIEW.md` for unresolved items.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os
import gc
from typing import Dict, List, Optional

import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_emotion_token_map,
    zero_shot_template,
    cleanup_memory,
)


# ============================================================
# Cache / environment
# ============================================================

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT


# ============================================================
# Config
# ============================================================

RUNS = [
    {
        "model_name": "google/gemma-2-2b",
        "layer_idx": 22,
    },
    {
        "model_name": "meta-llama/Llama-3.1-8B",
        "layer_idx": 26,
    },
]

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# Used to compute emotion vectors.
VECTOR_CSV_PATH = "data/self_reported_ekman_zeroshot_preds.csv"

# Used to calibrate alpha coefficients.
# Ideally this should be validation, not final test.
CALIBRATION_CSV_PATH = "data/self_reported_ekman_zeroshot_preds.csv"

TEXT_COL = "input_text"
LABEL_COL = "emotion"

DEVICE = "cuda"
DTYPE = torch.bfloat16
CACHE_DIR = None

BATCH_SIZE = 128
APPLY_PROMPT_TEMPLATE = False

VECTOR_MAX_EXAMPLES = None
CALIBRATION_MAX_EXAMPLES = None

# Candidate values for each alpha coefficient.
# Each emotion vector is unit-normalized, so these are residual-space step sizes.
ALPHA_GRID = [-8, -4, -2, -1, -0.5, 0, 0.5, 1, 2, 4, 8]

# Coordinate search passes.
N_SEARCH_PASSES = 1

OUTPUT_ROOT = "data/multivector_emotion_steering"


# ============================================================
# Global runtime objects
# ============================================================

llm = None
emotion_to_token = None
emotion_token_ids = None


# ============================================================
# Helpers
# ============================================================

def model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "-")


def load_emotion_df(
    csv_path: str,
    text_col: str = TEXT_COL,
    label_col: str = LABEL_COL,
    apply_prompt_template: bool = APPLY_PROMPT_TEMPLATE,
    max_examples: Optional[int] = None,
):
    df = pd.read_csv(csv_path)

    df = df[[text_col, label_col]].dropna().copy()
    df[label_col] = df[label_col].astype(str).str.lower().str.strip()
    df = df[df[label_col].isin(EMOTIONS)].copy()

    if max_examples is not None:
        df = df.iloc[:max_examples].copy()

    if apply_prompt_template:
        df["_prompt"] = df[text_col].apply(zero_shot_template)
    else:
        df["_prompt"] = df[text_col].astype(str)

    return df


def alpha_dict_to_string(alpha: Dict[str, float]) -> str:
    return ", ".join([f"{e}={alpha[e]:.4g}" for e in EMOTIONS])


def build_delta_vector(
    emotion_vectors: Dict[str, torch.Tensor],
    alpha: Dict[str, float],
) -> torch.Tensor:
    delta = None

    for emotion in EMOTIONS:
        coeff = float(alpha.get(emotion, 0.0))
        v = emotion_vectors[emotion].float()

        if delta is None:
            delta = coeff * v
        else:
            delta = delta + coeff * v

    return delta


# ============================================================
# Step 1: extract emotion vectors
# ============================================================

@torch.no_grad()
def collect_hidden_states_for_layer(
    dataset_csv_path: str,
    layer_idx: int,
    batch_size: int = BATCH_SIZE,
    text_col: str = TEXT_COL,
    label_col: str = LABEL_COL,
    apply_prompt_template: bool = APPLY_PROMPT_TEMPLATE,
    max_examples: Optional[int] = None,
):
    global llm

    df = load_emotion_df(
        csv_path=dataset_csv_path,
        text_col=text_col,
        label_col=label_col,
        apply_prompt_template=apply_prompt_template,
        max_examples=max_examples,
    )

    all_hidden = []
    all_labels = []

    for start in tqdm(
        range(0, len(df), batch_size),
        desc=f"Collecting hidden states at layer {layer_idx}",
    ):
        batch_df = df.iloc[start:start + batch_size]
        examples = batch_df["_prompt"].tolist()

        with llm.trace(examples):
            hidden = (
                llm.model.layers[layer_idx]
                .output[:, -1, :]
                .save()
            )

        all_hidden.append(hidden.detach().cpu().float())
        all_labels.extend(batch_df[label_col].tolist())

        cleanup_memory()

    H = torch.cat(all_hidden, dim=0)
    y = pd.Series(all_labels)

    return H, y


def compute_one_vs_rest_emotion_vectors(
    H: torch.Tensor,
    y: pd.Series,
    emotions: List[str] = EMOTIONS,
    eps: float = 1e-8,
):
    emotion_vectors = {}
    stats_rows = []

    for emotion in emotions:
        pos_mask = (y == emotion).values
        neg_mask = (y != emotion).values

        n_pos = int(pos_mask.sum())
        n_neg = int(neg_mask.sum())

        if n_pos == 0:
            raise ValueError(f"No positive examples found for emotion: {emotion}")
        if n_neg == 0:
            raise ValueError(f"No negative examples found for emotion: {emotion}")

        h_pos = H[pos_mask].mean(dim=0)
        h_neg = H[neg_mask].mean(dim=0)

        v = h_pos - h_neg
        raw_norm = float(v.norm().item())

        v = v / (v.norm() + eps)

        emotion_vectors[emotion] = v.cpu().float()

        stats_rows.append({
            "emotion": emotion,
            "n_pos": n_pos,
            "n_neg": n_neg,
            "raw_vector_norm": raw_norm,
        })

    stats_df = pd.DataFrame(stats_rows)

    return emotion_vectors, stats_df


# ============================================================
# Prediction with multi-vector steering
# ============================================================

@torch.no_grad()
def predict_batch_vanilla(examples: List[str]) -> List[str]:
    global llm, emotion_token_ids

    with llm.trace(examples):
        emotion_logits = llm.output.logits[:, -1, emotion_token_ids].save()

    pred_idx = emotion_logits.argmax(dim=-1).detach().cpu().tolist()
    return [EMOTIONS[i] for i in pred_idx]


@torch.no_grad()
def predict_batch_with_multivector_steering(
    examples: List[str],
    layer_idx: int,
    emotion_vectors: Dict[str, torch.Tensor],
    alpha: Dict[str, float],
) -> List[str]:
    global llm, emotion_token_ids

    if all(abs(float(alpha.get(e, 0.0))) < 1e-12 for e in EMOTIONS):
        return predict_batch_vanilla(examples)

    with llm.trace(examples):
        source_hidden = (
            llm.model.layers[layer_idx]
            .output[:, -1, :]
            .save()
        )

    delta = build_delta_vector(
        emotion_vectors=emotion_vectors,
        alpha=alpha,
    )

    delta = delta.to(
        device=source_hidden.device,
        dtype=source_hidden.dtype,
    )

    steered_hidden = source_hidden + delta.unsqueeze(0)

    with llm.trace(examples):
        llm.model.layers[layer_idx].output[:, -1, :] = steered_hidden
        emotion_logits = llm.output.logits[:, -1, emotion_token_ids].save()

    pred_idx = emotion_logits.argmax(dim=-1).detach().cpu().tolist()
    pred_emotions = [EMOTIONS[i] for i in pred_idx]

    cleanup_memory()

    return pred_emotions


# ============================================================
# Evaluation
# ============================================================

def evaluate_alpha(
    dataset_csv_path: str,
    layer_idx: int,
    emotion_vectors: Dict[str, torch.Tensor],
    alpha: Dict[str, float],
    batch_size: int = BATCH_SIZE,
    text_col: str = TEXT_COL,
    label_col: str = LABEL_COL,
    apply_prompt_template: bool = APPLY_PROMPT_TEMPLATE,
    max_examples: Optional[int] = None,
    return_predictions: bool = False,
):
    df = load_emotion_df(
        csv_path=dataset_csv_path,
        text_col=text_col,
        label_col=label_col,
        apply_prompt_template=apply_prompt_template,
        max_examples=max_examples,
    )

    preds = []

    for start in tqdm(
        range(0, len(df), batch_size),
        desc=f"Evaluating alpha: {alpha_dict_to_string(alpha)}",
    ):
        batch_df = df.iloc[start:start + batch_size]
        examples = batch_df["_prompt"].tolist()

        batch_preds = predict_batch_with_multivector_steering(
            examples=examples,
            layer_idx=layer_idx,
            emotion_vectors=emotion_vectors,
            alpha=alpha,
        )

        preds.extend(batch_preds)

    df["pred"] = preds
    df["correct"] = df["pred"] == df[label_col]

    y_true = df[label_col].tolist()
    y_pred = df["pred"].tolist()

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=EMOTIONS,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=EMOTIONS,
        average="weighted",
        zero_division=0,
    )

    acc = float(df["correct"].mean())

    report = classification_report(
        y_true,
        y_pred,
        labels=EMOTIONS,
        output_dict=True,
        zero_division=0,
    )

    summary = {
        "accuracy": acc,
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "n_examples": int(len(df)),
        **{f"alpha_{e}": float(alpha[e]) for e in EMOTIONS},
    }

    per_emotion_rows = []

    for emotion in EMOTIONS:
        emotion_mask = df[label_col] == emotion

        per_emotion_rows.append({
            "emotion": emotion,
            "n": int(emotion_mask.sum()),
            "precision": float(report[emotion]["precision"]),
            "recall": float(report[emotion]["recall"]),
            "f1": float(report[emotion]["f1-score"]),
            "accuracy_within_label": float(df.loc[emotion_mask, "correct"].mean()),
            **{f"alpha_{e}": float(alpha[e]) for e in EMOTIONS},
        })

    per_emotion_df = pd.DataFrame(per_emotion_rows)

    if return_predictions:
        return summary, per_emotion_df, df

    return summary, per_emotion_df


# ============================================================
# Step 2: calibrate alpha coefficients
# ============================================================

def coordinate_search_alpha(
    dataset_csv_path: str,
    layer_idx: int,
    emotion_vectors: Dict[str, torch.Tensor],
    alpha_grid: List[float] = ALPHA_GRID,
    n_passes: int = N_SEARCH_PASSES,
    batch_size: int = BATCH_SIZE,
    max_examples: Optional[int] = CALIBRATION_MAX_EXAMPLES,
):
    best_alpha = {emotion: 0.0 for emotion in EMOTIONS}

    best_summary, _ = evaluate_alpha(
        dataset_csv_path=dataset_csv_path,
        layer_idx=layer_idx,
        emotion_vectors=emotion_vectors,
        alpha=best_alpha,
        batch_size=batch_size,
        max_examples=max_examples,
    )

    best_score = best_summary["macro_f1"]

    history = [{
        "search_pass": 0,
        "searched_emotion": "baseline",
        "candidate_value": 0.0,
        "macro_f1": best_score,
        "accuracy": best_summary["accuracy"],
        "weighted_f1": best_summary["weighted_f1"],
        **{f"alpha_{e}": best_alpha[e] for e in EMOTIONS},
    }]

    print("\nInitial alpha:")
    print(best_alpha)
    print(f"Initial macro-F1: {best_score:.6f}")

    for search_pass in range(1, n_passes + 1):
        print("\n" + "#" * 80)
        print(f"Coordinate search pass {search_pass}/{n_passes}")
        print("#" * 80)

        for emotion in EMOTIONS:
            print("\n" + "-" * 80)
            print(f"Searching alpha for emotion direction: {emotion}")
            print("-" * 80)

            local_best_alpha = dict(best_alpha)
            local_best_score = best_score
            local_best_summary = best_summary

            for candidate_value in alpha_grid:
                trial_alpha = dict(best_alpha)
                trial_alpha[emotion] = float(candidate_value)

                summary, _ = evaluate_alpha(
                    dataset_csv_path=dataset_csv_path,
                    layer_idx=layer_idx,
                    emotion_vectors=emotion_vectors,
                    alpha=trial_alpha,
                    batch_size=batch_size,
                    max_examples=max_examples,
                )

                score = summary["macro_f1"]

                history.append({
                    "search_pass": search_pass,
                    "searched_emotion": emotion,
                    "candidate_value": float(candidate_value),
                    "macro_f1": score,
                    "accuracy": summary["accuracy"],
                    "weighted_f1": summary["weighted_f1"],
                    **{f"alpha_{e}": trial_alpha[e] for e in EMOTIONS},
                })

                print(
                    f"{emotion}={candidate_value}: "
                    f"macro-F1={score:.6f}, acc={summary['accuracy']:.6f}"
                )

                if score > local_best_score:
                    local_best_score = score
                    local_best_alpha = trial_alpha
                    local_best_summary = summary

            best_alpha = local_best_alpha
            best_score = local_best_score
            best_summary = local_best_summary

            print("\nCurrent best alpha:")
            print(best_alpha)
            print(f"Current best macro-F1: {best_score:.6f}")

    history_df = pd.DataFrame(history)

    return best_alpha, best_summary, history_df


# ============================================================
# Run
# ============================================================

all_final_summaries = []
all_final_per_emotion = []

for run in RUNS:
    MODEL_NAME = run["model_name"]
    LAYER_IDX = int(run["layer_idx"])

    model_slug = model_safe_name(MODEL_NAME)

    output_dir = os.path.join(
        OUTPUT_ROOT,
        model_slug,
        f"layer_{LAYER_IDX}",
    )
    os.makedirs(output_dir, exist_ok=True)

    print("\n" + "=" * 100)
    print(f"Loading model: {MODEL_NAME}")
    print(f"Target layer: {LAYER_IDX}")
    print("=" * 100)

    llm = load_llm(
        model_name=MODEL_NAME,
        device=DEVICE,
        dtype=DTYPE,
        cache_dir=CACHE_DIR,
    )

    emotion_to_token = build_emotion_token_map(
        llm=llm,
        emotions=EMOTIONS,
    )

    emotion_token_ids = [emotion_to_token[e] for e in EMOTIONS]

    # --------------------------------------------------------
    # 1. Extract emotion vectors
    # --------------------------------------------------------

    H, y = collect_hidden_states_for_layer(
        dataset_csv_path=VECTOR_CSV_PATH,
        layer_idx=LAYER_IDX,
        batch_size=BATCH_SIZE,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        apply_prompt_template=APPLY_PROMPT_TEMPLATE,
        max_examples=VECTOR_MAX_EXAMPLES,
    )

    emotion_vectors, vector_stats_df = compute_one_vs_rest_emotion_vectors(
        H=H,
        y=y,
        emotions=EMOTIONS,
    )

    vector_path = os.path.join(output_dir, "emotion_vectors.pt")
    vector_stats_path = os.path.join(output_dir, "emotion_vector_stats.csv")

    torch.save(emotion_vectors, vector_path)
    vector_stats_df.to_csv(vector_stats_path, index=False)

    print(f"Saved emotion vectors to: {vector_path}")
    print(f"Saved vector stats to: {vector_stats_path}")

    del H
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 2. Calibrate alpha coefficients
    # --------------------------------------------------------

    best_alpha, best_summary, search_history_df = coordinate_search_alpha(
        dataset_csv_path=CALIBRATION_CSV_PATH,
        layer_idx=LAYER_IDX,
        emotion_vectors=emotion_vectors,
        alpha_grid=ALPHA_GRID,
        n_passes=N_SEARCH_PASSES,
        batch_size=BATCH_SIZE,
        max_examples=CALIBRATION_MAX_EXAMPLES,
    )

    search_history_path = os.path.join(output_dir, "alpha_search_history.csv")
    search_history_df.to_csv(search_history_path, index=False)

    best_alpha_df = pd.DataFrame([{
        "model_name": MODEL_NAME,
        "layer_idx": LAYER_IDX,
        **{f"alpha_{e}": best_alpha[e] for e in EMOTIONS},
        **best_summary,
    }])

    best_alpha_path = os.path.join(output_dir, "best_alpha.csv")
    best_alpha_df.to_csv(best_alpha_path, index=False)

    print(f"Saved alpha search history to: {search_history_path}")
    print(f"Saved best alpha to: {best_alpha_path}")

    # --------------------------------------------------------
    # 3. Final evaluation with best alpha
    # --------------------------------------------------------

    final_summary, final_per_emotion, final_predictions = evaluate_alpha(
        dataset_csv_path=CALIBRATION_CSV_PATH,
        layer_idx=LAYER_IDX,
        emotion_vectors=emotion_vectors,
        alpha=best_alpha,
        batch_size=BATCH_SIZE,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        apply_prompt_template=APPLY_PROMPT_TEMPLATE,
        max_examples=CALIBRATION_MAX_EXAMPLES,
        return_predictions=True,
    )

    final_summary["model_name"] = MODEL_NAME
    final_summary["layer_idx"] = LAYER_IDX

    final_per_emotion["model_name"] = MODEL_NAME
    final_per_emotion["layer_idx"] = LAYER_IDX

    final_predictions["model_name"] = MODEL_NAME
    final_predictions["layer_idx"] = LAYER_IDX
    for emotion in EMOTIONS:
        final_predictions[f"alpha_{emotion}"] = best_alpha[emotion]

    final_summary_path = os.path.join(output_dir, "final_summary.csv")
    final_per_emotion_path = os.path.join(output_dir, "final_per_emotion.csv")
    final_predictions_path = os.path.join(output_dir, "final_predictions.csv")

    pd.DataFrame([final_summary]).to_csv(final_summary_path, index=False)
    final_per_emotion.to_csv(final_per_emotion_path, index=False)
    final_predictions.to_csv(final_predictions_path, index=False)

    print(f"Saved final summary to: {final_summary_path}")
    print(f"Saved final per-emotion results to: {final_per_emotion_path}")
    print(f"Saved final predictions to: {final_predictions_path}")

    all_final_summaries.append(pd.DataFrame([final_summary]))
    all_final_per_emotion.append(final_per_emotion)

    del llm
    del emotion_vectors
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Save aggregate results
# ============================================================

if len(all_final_summaries) > 0:
    aggregate_summary = pd.concat(all_final_summaries, ignore_index=True)
    aggregate_summary_path = os.path.join(
        OUTPUT_ROOT,
        "all_models_multivector_steering_summary.csv",
    )
    aggregate_summary.to_csv(aggregate_summary_path, index=False)
    print(f"Saved aggregate summary to: {aggregate_summary_path}")

if len(all_final_per_emotion) > 0:
    aggregate_per_emotion = pd.concat(all_final_per_emotion, ignore_index=True)
    aggregate_per_emotion_path = os.path.join(
        OUTPUT_ROOT,
        "all_models_multivector_steering_per_emotion.csv",
    )
    aggregate_per_emotion.to_csv(aggregate_per_emotion_path, index=False)
    print(f"Saved aggregate per-emotion results to: {aggregate_per_emotion_path}")

print("\nDone.")